# Étude d'ablation des termes physiques -- filamentation femtoseconde dans SiO2

Ce notebook pilote `sim/filament_sim.py` (solveur split-step Hankel/FFT, RK4,
kernel CUDA pour les équations de taux) avec **six interrupteurs indépendants**
sur l'équation de propagation du champ (éq. 3) :

| Interrupteur | Terme physique |
|---|---|
| `enable_kerr_instantaneous` | Kerr électronique instantané, `(1-f_R)\|E\|^2 E` |
| `enable_kerr_raman` | Kerr Raman/moléculaire retardé, `f_R (R * \|E\|^2) E` |
| `enable_self_steepening` | opérateur `T-hat = 1 + (i/omega0) d/dt` (auto-raidissement) |
| `enable_photoionization_loss` | perte d'énergie du champ par photoionisation |
| `enable_plasma_defocusing` | terme de phase induit par le plasma (défocalisation) |
| `enable_plasma_absorption` | absorption par Bremsstrahlung inverse |

Ces interrupteurs n'agissent QUE sur le champ. La génération de porteurs
(éq. 6-7 : avalanche, recombinaison, canal STE) reste pilotée par
`enable_avalanche` / `enable_recombination` / `enable_ste`, déjà présents dans
le solveur -- désactiver par ex. `enable_photoionization_loss` retire le terme
de perte du champ mais PAS la génération d'électrons libres par le même taux
de Keldysh (qui continue d'alimenter `rho`). C'est voulu : ça permet
d'isoler "qu'est-ce que ce terme fait au faisceau" de "combien d'électrons
sont créés".

Structure du notebook :
0. Fonctions de tracé (réutilisées partout dans le notebook)
1. **Vérification d'abord** : reproduire Couairon et al., PRB 2005 (800 nm)
   avec les paramètres exacts de l'article, pour valider le solveur sur un
   résultat publié avant de lancer quoi que ce soit sur ta propre expérience
2. Paramètres de base de TON expérience (grille + matériau repris de l'article, STE actif)
3. Interrupteurs interactifs + lancement manuel d'une configuration
4. Validation initiale : lancer `full` seul et vérifier sa cohérence
5. Boucle d'ablation automatique sur les 7 scénarios restants -- figure
   compacte à la volée après chacun
6. Comparaison superposée des scénarios (figures 7/8/9/13 style)
7. Tableau récapitulatif comparatif
8. Export vers la page web Abel-transform (`web/abel_phase_explorer.py`)

**Prérequis** : GPU NVIDIA + CUDA + `cupy`. Ce notebook a été écrit et relu
attentivement mais n'a PAS pu être exécuté dans l'environnement de rédaction
(pas de GPU disponible) -- à tester sur ta machine avant de lancer la boucle
complète sur une grille fine.

## Installation des dépendances

À exécuter en premier. Le post-traitement ne demande que numpy / scipy / matplotlib ; `cupy` n'est nécessaire que pour lancer un nouveau calcul sur GPU.


In [ ]:
# [deps-install-cell]
# ============================================================================
#  Installation des dependances
# ============================================================================
# A executer une fois, en premier. Si cupy est installe par cette cellule,
# REDEMARRER LE NOYAU avant de continuer.
#
# Deux niveaux :
#   - CPU  : numpy / scipy / matplotlib (+ tqdm / ipywidgets). Suffisent pour tout
#            le POST-TRAITEMENT, c'est-a-dire relire un result.npz deja calcule
#            et regenerer les figures.
#   - GPU  : cupy. Necessaire uniquement pour LANCER un calcul
#            (filament_sim.run()), qui est un solveur CUDA.
import importlib
import importlib.util
import re
import shutil
import subprocess
import sys

REQUIRED = ["numpy", "scipy", "matplotlib", "pillow"]
REQUIRED += ['tqdm', 'ipywidgets']
_IMPORT_NAME = {"pillow": "PIL", "ipywidgets": "ipywidgets"}


def _pip(*args):
    print("  pip install", *args)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


print("Dependances CPU :")
for pkg in REQUIRED:
    mod = _IMPORT_NAME.get(pkg, pkg)
    if importlib.util.find_spec(mod) is None:
        _pip(pkg)
    else:
        print(f"  {pkg:12s} deja present")

print("\nDependance GPU (cupy) :")
if importlib.util.find_spec("cupy") is not None:
    import cupy
    print(f"  cupy {cupy.__version__} deja present")
    try:
        print(f"  GPU visible : {cupy.cuda.runtime.getDeviceCount()} device(s)")
    except Exception as exc:
        print(f"  /!\\ cupy importe mais aucun GPU utilisable ({type(exc).__name__})")
else:
    # La roue cupy depend de la version de CUDA du pilote : il n'existe pas de
    # paquet "cupy" generique qui marche partout, d'ou la detection.
    cuda_major = None
    if shutil.which("nvidia-smi"):
        try:
            out = subprocess.check_output(["nvidia-smi"], text=True)
            m = re.search(r"CUDA Version:\s*(\d+)\.", out)
            cuda_major = int(m.group(1)) if m else None
        except Exception:
            pass
    if cuda_major is None:
        print("  aucun GPU NVIDIA detecte -> cupy N'EST PAS installe.")
        print("  Le post-traitement fonctionne quand meme sur un result.npz")
        print("  deja calcule ; seul filament_sim.run() a besoin du GPU.")
    else:
        _pip(f"cupy-cuda{12 if cuda_major >= 12 else 11}x")
        print("  cupy installe -> REDEMARRER LE NOYAU avant de continuer.")

print("\nVersions :")
for _m in ("numpy", "scipy", "matplotlib"):
    try:
        print(f"  {_m:12s} {importlib.import_module(_m).__version__}")
    except ImportError:
        print(f"  {_m:12s} ABSENT")


In [ ]:
# !pip install numpy scipy matplotlib ipywidgets tqdm cupy-cuda12x --break-system-packages
import sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from tqdm.auto import tqdm

# --- localisation des modules du depot ------------------------------------
# Ce notebook a besoin de l'arborescence "deliverable (3)/{sim,web,notebooks}",
# pas seulement du fichier .ipynb. On cherche sim/ depuis le repertoire courant
# puis en remontant, ce qui marche que le noyau soit lance depuis notebooks/,
# depuis la racine du depot, ou ailleurs.
SIM_DIR = None   # <- mettre ici le chemin de sim/ si la recherche echoue


def _resolve_pkg_dirs(explicit=None):
    roots = []
    if explicit:
        p = Path(explicit).expanduser().resolve()
        roots.append(p.parent if p.name == "sim" else p)
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        roots += [base, base / "deliverable (3)",
                  base / ".github" / "deliverable (3)"]
    for root in roots:
        if (root / "sim" / "filament_sim.py").is_file():
            return {n: root / n for n in ("sim", "web") if (root / n).is_dir()}
    return {}


_dirs = _resolve_pkg_dirs(SIM_DIR)
if not _dirs:
    raise ModuleNotFoundError(
        "dossier sim/ introuvable."
        "\n  repertoire courant : " + str(Path.cwd()) +
        "\n  Ce notebook a besoin de l'arborescence complete du depot, pas"
        "\n  seulement du .ipynb. Recupere le dossier 'deliverable (3)' entier"
        "\n  (sim/, web/, notebooks/) et lance le noyau depuis"
        "\n  'deliverable (3)/notebooks/', ou renseigne SIM_DIR ci-dessus.")
for _n, _d in _dirs.items():
    sys.path.insert(0, str(_d))
print("modules du depot :", _dirs["sim"])

from filament_sim import run, FIELD_TOGGLES, n_sellmeier, code_fingerprint

print("FIELD_TOGGLES:", FIELD_TOGGLES)

In [ ]:
OUT_ROOT = Path("runs_ablation")
OUT_ROOT.mkdir(exist_ok=True)

# Rempli au fur et à mesure par la section 1 (vérification Couairon 2005) et
# la section 4 (validation de TON expérience), donc défini une seule fois ici
# -- aucune des deux ne doit faire `results = {...}` (ça écraserait l'autre).
results = {}

def load_scenario_npz(out_dir, ignore_stale=False):
    """Relit un result.npz deja calcule, SAUF s'il a ete produit par une
    version anterieure du solveur.

    Chaque run inscrit dans params.json une empreinte du code source
    (filament_sim.py + keldysh.py). Si elle ne correspond plus, ce cache est
    perime : la fonction renvoie None, ce qui declenche automatiquement un
    recalcul. Plus besoin de penser a vider les dossiers a la main apres un
    changement de physique. `ignore_stale=True` force la relecture quand meme.

    (Il reste une chose que ceci ne peut pas faire : si le kernel Jupyter a
    deja importe filament_sim, editer le .py ne change rien tant que le kernel
    n'est pas redemarre. L'empreinte est calculee a partir du fichier sur
    disque, donc elle detecte aussi ce cas et te le signale.)"""
    npz_path = Path(out_dir) / "result.npz"
    if not npz_path.exists():
        return None
    pj = Path(out_dir) / "params.json"
    if not ignore_stale and pj.exists():
        try:
            old_fp = json.loads(pj.read_text()).get("code_fingerprint")
        except Exception:
            old_fp = None
        new_fp = code_fingerprint()
        if old_fp != new_fp:
            print(f"[{Path(out_dir).name}] cache PERIME "
                  f"(empreinte {old_fp} != {new_fp}) -> recalcul")
            return None
    try:
        return dict(np.load(npz_path, allow_pickle=True))
    except Exception as exc:
        # Un run tue en cours d'ecriture (disque plein -> OSError errno 5)
        # laissait un npz tronque ; le relire plus tard levait un zlib
        # "invalid block type" et cassait tout le notebook. On le traite
        # comme un cache absent : le scenario est simplement recalcule.
        print(f"[{Path(out_dir).name}] result.npz ILLISIBLE ({type(exc).__name__}: {exc}) "
              f"-> ignore, recalcul")
        return None

## 0. Fonctions de tracé

Définies ICI (avant la boucle) pour deux usages :
- un tracé **compact à la volée** juste après chaque scénario dans la boucle
  d'ablation (section 4), pour débugger en direct ;
- un tracé **superposé** de tous les scénarios une fois la boucle terminée
  (section 6), pour comparer l'effet de chaque terme.

Les fonctions `_plot_*_ax(ax, ...)` contiennent la logique de tracé sur un
`Axes` donné ; les fonctions publiques (`plot_fig8_...`, `plot_scenario_summary`,
...) les réutilisent pour éviter de dupliquer le code entre les deux usages.

In [ ]:
def _z_um(res, z_shift_um=0.0):
    """z_shift_um permet de replacer l'origine sur la face d'entree (comme
    l'article, z=0 = entree) au lieu de l'origine du solveur (z=0 = foyer
    lineaire). N'affecte que l'affichage -- z_target_um reste toujours
    exprime dans le repere du solveur (centre sur le foyer)."""
    return np.asarray(res["z"]) * 1e6 + z_shift_um

def _r_um(res):
    r = np.asarray(res["r"])
    return r * 1e6 if np.max(np.abs(r)) < 1e-3 else r

def _onaxis_index(res):
    return int(np.argmin(np.abs(np.asarray(res["r"]))))

def _peak_z_um(res):
    """z (µm) où l'intensité crête on-axis est maximale -- même critère que
    figures_article._iz_of(z_um=None), pour que 'quel z regarder' soit
    cohérent partout au lieu d'un nombre codé en dur par figure."""
    z_um = _z_um(res)
    return float(z_um[int(np.argmax(res["Imax_z"]))])

def _plot_peak_intensity_ax(ax, res, label, I_clamp=5e13, show_clamp=True, z_shift_um=0.0,
                             xlim=None, ylim=None):
    ax.plot(_z_um(res, z_shift_um), res["Imax_z"], lw=1.6, label=label)
    if show_clamp:
        ax.axhline(I_clamp, ls="--", color="crimson", lw=1, label=f"I_clamp≈{I_clamp:.0e}")
    ax.set_yscale("log")
    if xlim is not None: ax.set_xlim(*xlim)
    if ylim is not None: ax.set_ylim(*ylim)
    ax.set_xlabel("z (µm)"); ax.set_ylabel("Peak intensity (W/cm²)")
    ax.set_title("Peak intensity vs z")

def _plot_electron_density_ax(ax, res, label, rho_lines=(1e20, 2e20),
                               nc_probe_cm3=None, show_lines=True, z_shift_um=0.0,
                               xlim=None, ylim=(1e15, 1e21)):
    i_axis = _onaxis_index(res)
    rho_onaxis = res["rho_rz"][:, i_axis]
    ax.plot(_z_um(res, z_shift_um), np.clip(rho_onaxis, 1e-3, None), lw=1.6, label=label)
    if show_lines:
        for rl in rho_lines:
            ax.axhline(rl, ls="--", color="orange" if rl == max(rho_lines) else "crimson", lw=1)
        if nc_probe_cm3 is not None:
            ax.axhline(nc_probe_cm3, ls=":", color="purple", lw=1,
                       label=f"n_c(probe)={nc_probe_cm3:.2e}")
    ax.set_yscale("log")
    if xlim is not None: ax.set_xlim(*xlim)
    if ylim is not None: ax.set_ylim(*ylim)
    ax.set_xlabel("z (µm)"); ax.set_ylabel("On-axis ρ_e (cm⁻³)")
    ax.set_title("On-axis electron density vs z")

def _plot_free_vs_trapped_ax(ax, res, z_target_um, label="", z_shift_um=0.0,
                              xlim=None, ylim=(1e16, 1e22), ylim_I=None):
    # Prefer the full-time-resolution on-axis trace (rho_onaxis_t/I_onaxis_t,
    # r index 0 only -- see filament_sim.py Integrator._record) over the
    # rho_t_stride-subsampled rho_rzt/I_rzt cube: the multiphoton rate is
    # very sensitive to intensity (~I^K), so a narrow spike missed between
    # two saved samples in the coarse cube can make this look wrong even
    # though the CUDA kernel (full grid) computed it correctly. Same fix as
    # figures_article.fig2_populations.
    # z_target_um est toujours dans le repere du solveur (foyer = 0) --
    # z_shift_um ne sert qu'a l'affichage du titre (repere article, entree = 0).
    z_um = _z_um(res)
    iz = int(np.argmin(np.abs(z_um - z_target_um)))
    ir = _onaxis_index(res)

    if res.get("rho_onaxis_t") is not None and res.get("t_full_fs") is not None:
        t_fs  = np.asarray(res["t_full_fs"])
        rho_e = res["rho_onaxis_t"][iz, :]
        rho_s = res["rho_s_onaxis_t"][iz, :]
        I_t   = res["I_onaxis_t"][iz, :]
    elif res.get("rho_rzt") is not None and np.asarray(res["rho_rzt"]).shape != ():
        t_fs  = np.asarray(res["t_sub_fs"])
        rho_e = res["rho_rzt"][iz, ir, :]
        rho_s = res["rho_s_rzt"][iz, ir, :]
        I_t   = res["I_rzt"][iz, ir, :]
    else:
        ax.text(0.5, 0.5, "rho_rzt/rho_onaxis_t indisponible\n(relance avec rho_t_stride > 0)",
                ha="center", va="center", fontsize=9)
        ax.set_axis_off()
        return

    # Floor at ~0 (not at ylim[0]=1e16) so points genuinely below the visible
    # range (pre-ionization rho_e~0, or rho_s~0 entirely when enable_ste=False)
    # fall OUTSIDE the plotted window instead of being dragged up onto a fake
    # flat plateau at the old 1e16 floor, which used to sit inside ylim and
    # get drawn as if it were real (near-)threshold data.
    ax.plot(t_fs, np.clip(rho_e, 1e-30, None), color="black", lw=1.4, label="ρ_e libre")
    ax.plot(t_fs, np.clip(rho_s, 1e-30, None), color="tab:blue", lw=1.4, label="ρ_s piégé")
    ax.set_yscale("log")
    if xlim is not None: ax.set_xlim(*xlim)
    if ylim is not None: ax.set_ylim(*ylim)
    ax.set_xlabel("Time (fs)"); ax.set_ylabel("ρ (cm⁻³)")
    ax.set_title(f"Free vs trapped @ z={z_um[iz] + z_shift_um:.0f}µm  [{label}]")

    ax2 = ax.twinx()
    ax2.plot(t_fs, I_t, "--", color="crimson", lw=1.0, label="Pulse intensity")
    ax2.set_yscale("log")
    if ylim_I is not None: ax2.set_ylim(*ylim_I)
    ax2.set_ylabel("Intensity (W/cm²)", color="crimson")
    ax2.tick_params(axis="y", colors="crimson")

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=7)

def _plot_fluence_contours_ax(ax, res, levels=(0.1, 1.0, 5.0, 10.0), label="", z_shift_um=0.0,
                               xlim=None, rlim=None):
    z_um = _z_um(res, z_shift_um)
    r_um = _r_um(res)
    fluence_Jcm2 = res["fluence_rz"]  # deja en J/cm^2 (voir filament_sim.py: invE2 inclut le facteur m^2->cm^2)

    cs = ax.contour(z_um, r_um, fluence_Jcm2.T, levels=levels, colors="black", linewidths=0.8)
    ax.clabel(cs, inline=True, fontsize=6, fmt="%.1f J/cm²")

    # FWHM/2 : demi-largeur où la fluence radiale tombe à moitié du max, à chaque z.
    # On prend la PREMIERE traversée en partant de l'axe (avec interpolation
    # linéaire), pas la dernière : pendant la phase de défocalisation par le
    # plasma le profil radial développe des anneaux, et un anneau externe
    # repassant au-dessus de la mi-hauteur faisait sauter l'ancienne version
    # (`above[-1]`, le point le PLUS externe) très loin sur le bord -- ce qui
    # déformait la courbe en tirets exactement dans la zone 60-100 µm où la
    # Fig. 7 de l'article est intéressante.
    half = 0.5 * fluence_Jcm2.max(axis=1)
    fwhm_half_um = np.full(len(z_um), np.nan)
    Nr_pos = len(r_um) // 2
    r_pos = r_um[Nr_pos:]
    for iz in range(len(z_um)):
        prof = fluence_Jcm2[iz, Nr_pos:]
        below = np.where(prof < half[iz])[0]
        if below.size and below[0] > 0:
            j = below[0]
            f0, f1 = prof[j - 1], prof[j]
            w = 0.0 if f0 == f1 else (f0 - half[iz]) / (f0 - f1)
            fwhm_half_um[iz] = r_pos[j - 1] + w * (r_pos[j] - r_pos[j - 1])
    ax.plot(z_um, fwhm_half_um, "--", color="tab:blue", lw=1, label="Beam radius (FWHM/2)")
    ax.plot(z_um, -fwhm_half_um, "--", color="tab:blue", lw=1)

    if xlim is not None: ax.set_xlim(*xlim)
    if rlim is not None: ax.set_ylim(*rlim)
    ax.set_xlabel("z (µm)"); ax.set_ylabel("r (µm)")
    ax.set_title(f"Fluence contours and beam FWHM vs z  [{label}]")

def _plot_energy_losses_ax(ax, res, label="", z_shift_um=0.0, xlim=(0, 150), ylim=(1e-3, 1e0)):
    """Fig. 12 style -- pertes d'energie cumulees (fraction de l'energie
    d'entree U0_uJ) vs z : Plasma+Photo (continu), Plasma seul (tirete),
    Photo seul (point-tirete) -- memes styles que la Fig. 12 de Couairon 2005."""
    z_um = _z_um(res, z_shift_um)
    tag = f"{label} -- " if label else ""
    ax.semilogy(z_um, np.clip(res["E_total_z"],  1e-6, None), "-",  color="black", lw=1.8, label=f"{tag}Plasma+Photo")
    ax.semilogy(z_um, np.clip(res["E_plasma_z"], 1e-6, None), "--", color="black", lw=1.4, label=f"{tag}Plasma")
    ax.semilogy(z_um, np.clip(res["E_MPI_z"],    1e-6, None), "-.", color="black", lw=1.4, label=f"{tag}Photo")
    if xlim is not None: ax.set_xlim(*xlim)
    if ylim is not None: ax.set_ylim(*ylim)
    ax.set_xlabel("z (µm)"); ax.set_ylabel("Energy losses (fraction of U0)")
    ax.set_title("Energy losses vs z")

def fluence_level_extent(res, levels=(1.0, 2.0, 3.0), z_shift_um=0.0, label=""):
    """Étendue en z du domaine où la fluence ON-AXIS dépasse chaque niveau.

    C'est le test chiffré que l'article donne lui-même pour la Fig. 7 (Sec. V) :
      « for an incident pulse energy of 0.45 µJ, the domain where the flux
        exceeds 1 J/cm2 extends from 45 to 90 µm and, for a 1.1 µJ pulse,
        it extends from 25 to 110 µm »
    -> compare des nombres au lieu de comparer des formes à l'oeil."""
    z_um = _z_um(res, z_shift_um)
    ia = _onaxis_index(res)
    f_axis = np.asarray(res["fluence_rz"])[:, ia]
    print(f"[{label}] fluence on-axis max = {f_axis.max():.2f} J/cm² "
          f"@ z = {z_um[int(np.argmax(f_axis))]:.0f} µm")
    out = {}
    for lv in levels:
        m = f_axis >= lv
        if m.any():
            out[lv] = (float(z_um[m][0]), float(z_um[m][-1]))
            print(f"    >= {lv:g} J/cm² : z de {out[lv][0]:6.1f} à {out[lv][1]:6.1f} µm"
                  f"   (longueur {out[lv][1]-out[lv][0]:.0f} µm)")
        else:
            out[lv] = None
            print(f"    >= {lv:g} J/cm² : JAMAIS atteint")
    return out

def run_health_check(res, out_dir=None, label=""):
    """Confronte un run aux valeurs chiffrees de Couairon 2005, et rappelle
    quels interrupteurs ont REELLEMENT servi (lus dans params.json).

    Le dernier point est le plus important en pratique : `load_scenario_npz`
    relit un result.npz existant sans regarder la version du code qui l'a
    produit, donc un run mis en cache peut faire croire qu'un changement de
    solveur n'a rien change. Si `enable_spectral_filter` n'apparait pas, le
    npz vient d'une version anterieure -> supprimer le dossier et relancer."""
    print(f"=== {label} ===")

    # 1) intensite crete -- article : 5 +/- 0.5e13 W/cm2, DECROISSANTE avec l'energie
    I_pk = float(np.max(res["Imax_z"]))
    z_pk = _z_um(res)[int(np.argmax(res["Imax_z"]))]
    flag = "OK" if 4.5e13 <= I_pk <= 5.5e13 else "HORS BANDE"
    print(f"  I_max            = {I_pk:.3e} W/cm2 @ z_sim={z_pk:+.0f} um   [article 5+/-0.5e13 -> {flag}]")

    # 2) densite electronique -- article : clampee entre 2e20 et 4e20 cm-3
    rho_pk = float(np.max(res["rho_rz"]))
    flag = "OK" if 2e20 <= rho_pk <= 4e20 else "HORS BANDE"
    print(f"  rho_e max        = {rho_pk:.3e} cm-3   [article 2-4e20 -> {flag}]")
    if rho_pk > 0.5 * 2.1e22:
        print("    /!\\ rho_e s'approche de rho_max=2.1e22 : emballement, pas un clampage physique")

    # 3) pertes d'energie cumulees (Fig. 12) et transmission (Fig. 5)
    if "E_total_z" in res and np.max(res["E_total_z"]) > 0:
        loss = float(np.max(res["E_total_z"]))
        print(f"  pertes totales   = {loss*100:.1f} %  ->  transmission {100*(1-loss):.1f} %   [cf. Fig. 5]")

    # 4) quelle version du solveur a produit ce npz
    if out_dir is not None:
        pj = Path(out_dir) / "params.json"
        if pj.exists():
            prm = json.loads(pj.read_text())
            tg = prm.get("toggles", {})
            sf = tg.get("enable_spectral_filter", prm.get("enable_spectral_filter"))
            if sf is None:
                print("  filtre spectral  : ABSENT de params.json -> npz produit par une version")
                print("                     ANTERIEURE du solveur. Supprime le dossier et relance :")
                print(f"                       import shutil; shutil.rmtree(r'{out_dir}')")
            else:
                print(f"  filtre spectral  : enable_spectral_filter = {sf}")
            stf = tg.get("enable_space_time_focusing", prm.get("enable_space_time_focusing"))
            print(f"  space-time focus : enable_space_time_focusing = {stf}")
        else:
            print(f"  (params.json introuvable dans {out_dir})")

In [ ]:
def plot_fig8_peak_intensity(results, scenario_names=None, I_clamp=5e13, save=None, z_shift_um=0.0,
                              xlim=None, ylim=None):
    """Figure 8 style -- intensité crête vs z, superposée sur plusieurs scénarios."""
    scenario_names = scenario_names or list(results.keys())
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, name in enumerate(scenario_names):
        _plot_peak_intensity_ax(ax, results[name], name, I_clamp=I_clamp, show_clamp=(i == 0),
                                 z_shift_um=z_shift_um, xlim=xlim, ylim=ylim)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=150)
    return fig

def plot_fig9_electron_density(results, scenario_names=None, rho_lines=(1e20, 2e20),
                                nc_probe_cm3=None, save=None, z_shift_um=0.0,
                                xlim=None, ylim=(1e15, 1e21)):
    """Figure 9 style -- densité électronique on-axis vs z, superposée sur plusieurs scénarios."""
    scenario_names = scenario_names or list(results.keys())
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, name in enumerate(scenario_names):
        _plot_electron_density_ax(ax, results[name], name, rho_lines=rho_lines,
                                   nc_probe_cm3=nc_probe_cm3, show_lines=(i == 0),
                                   z_shift_um=z_shift_um, xlim=xlim, ylim=ylim)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=150)
    return fig

def plot_fig13_free_vs_trapped(res, z_target_um=None, label="", save=None, z_shift_um=0.0,
                                xlim=None, ylim=(1e16, 1e22), ylim_I=None):
    """Figure 13 style -- électrons libres vs piégés + intensité du pulse, à z fixé.
    z_target_um=None -> z du pic d'intensité on-axis (même critère que
    figures_article.fig2_populations), au lieu d'un nombre codé en dur.
    z_target_um reste dans le repere du solveur (foyer=0) meme si z_shift_um != 0."""
    if z_target_um is None:
        z_target_um = _peak_z_um(res)
    fig, ax = plt.subplots(figsize=(9, 6))
    _plot_free_vs_trapped_ax(ax, res, z_target_um, label=label, z_shift_um=z_shift_um,
                              xlim=xlim, ylim=ylim, ylim_I=ylim_I)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=150)
    return fig

def plot_fig7_fluence_contours(res, levels=(0.1, 1.0, 5.0, 10.0), label="", save=None, z_shift_um=0.0,
                                xlim=None, rlim=None):
    """Figure 7 style -- contours de fluence + enveloppe FWHM/2 vs z."""
    fig, ax = plt.subplots(figsize=(9, 3.5))
    _plot_fluence_contours_ax(ax, res, levels=levels, label=label, z_shift_um=z_shift_um,
                               xlim=xlim, rlim=rlim)
    ax.legend(fontsize=8)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=150)
    return fig

def plot_scenario_summary(res, name, z_target_um=None, I_clamp=5e13,
                           rho_lines=(1e20, 2e20), nc_probe_cm3=None,
                           fluence_levels=(0.1, 1.0, 5.0, 10.0), save=None, z_shift_um=0.0,
                           z_xlim=None, I_ylim=None, rho_ylim=(1e15, 1e21),
                           trapped_ylim=(1e16, 1e22), trapped_ylim_I=None,
                           fluence_xlim=None, fluence_rlim=None):
    """
    Figure compacte 2x2 (une seule figure) pour UN scénario -- appelée à la
    volée juste après chaque simulation dans la boucle d'ablation, pour
    débugger en direct (vérifier que 'full' a une allure physique normale
    sans attendre les 7 autres scénarios).
    z_target_um=None -> z du pic d'intensité on-axis (voir plot_fig13_free_vs_trapped).
    z_shift_um : decale l'axe z affiche (ex. +75 pour repasser au repere de
    l'article, entree=0, si le solveur utilise foyer=0). Ne change pas
    z_target_um, qui reste dans le repere du solveur.
    z_xlim/I_ylim/rho_ylim/trapped_ylim/trapped_ylim_I/fluence_xlim/fluence_rlim :
    bornes explicites (repere article) pour matcher exactement les Figs. 8/9/10/7
    de Couairon 2005 -- laisser a None garde l'auto-echelle (etude d'ablation).
    """
    if z_target_um is None:
        z_target_um = _peak_z_um(res)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    fig.suptitle(f"Scenario: {name}  (z_target = {z_target_um + z_shift_um:+.0f} µm)",
                 fontsize=13, fontweight="bold")

    _plot_peak_intensity_ax(axes[0, 0], res, name, I_clamp=I_clamp, z_shift_um=z_shift_um,
                             xlim=z_xlim, ylim=I_ylim)
    axes[0, 0].legend(fontsize=7)

    _plot_electron_density_ax(axes[0, 1], res, name, rho_lines=rho_lines, nc_probe_cm3=nc_probe_cm3,
                               z_shift_um=z_shift_um, xlim=z_xlim, ylim=rho_ylim)
    axes[0, 1].legend(fontsize=7)

    _plot_free_vs_trapped_ax(axes[1, 0], res, z_target_um, label=name, z_shift_um=z_shift_um,
                              ylim=trapped_ylim, ylim_I=trapped_ylim_I)

    _plot_fluence_contours_ax(axes[1, 1], res, levels=fluence_levels, label=name, z_shift_um=z_shift_um,
                               xlim=fluence_xlim, rlim=fluence_rlim)
    axes[1, 1].legend(fontsize=7)

    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=150)
    plt.show()
    return fig

def plot_fig12_energy_losses(res, label="", save=None, z_shift_um=0.0, xlim=(0, 150), ylim=(1e-3, 1e0)):
    """Figure 12 style -- pertes d'energie cumulees (Plasma/Photo/combine) vs z."""
    fig, ax = plt.subplots(figsize=(7.5, 5))
    _plot_energy_losses_ax(ax, res, label=label, z_shift_um=z_shift_um, xlim=xlim, ylim=ylim)
    ax.legend(fontsize=8)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=150)
    return fig

## 1. Vérification : reproduire Couairon et al., PRB 71, 125435 (2005)

Ce papier (dont sont issus les styles des figures 7/8/9 utilisées plus haut)
donne des paramètres numériques précis -- différents de ceux de
`BASE_PARAMS` (les tiens, à 1030 nm). Pour vérifier que le solveur reproduit
bien leurs résultats publiés (Figs. 6, 7, 8, 9, 12, 13 du papier), on relance
avec **leurs** paramètres exacts, tirés du texte :

| Paramètre | Papier (Couairon 2005) | `BASE_PARAMS` (ton expérience) |
|---|---|---|
| longueur d'onde | 800 nm | 1030 nm |
| durée FWHM | 160 fs (t_p = 136 fs) | 263 fs |
| waist au foyer w_f (mesuré dans l'air, objectif 20×, NA=0.5) | 1.1 µm | 10 µm |
| début de boîte | d = 75 µm avant le foyer | via `z_focus_air_um` |
| énergie | 1.1 µJ (Figs. 6, 7b, 10, 12, 13) | 15 µJ |
| n2 | 3.54e-16 cm²/W = **3.54e-20 m²/W** | 2.4e-20 |
| τ_c (collision) | **1e-14 s** | 1.7e-15 s (défaut du solveur) |
| τ_r (recombinaison) | **150 fs** | 330 fs |
| canal STE (auto-piégeage) | absent du modèle du papier | actif |

**Point d'attention** : `BASE_PARAMS` utilise `n2=2.4e-20`, alors que la
valeur mesurée dans cet article est `3.54e-20` (n2 de la silice varie peu
entre 800 et 1030 nm) -- vérifie si `2.4e-20` est bien voulu pour ton
expérience, ou si c'est une valeur à corriger.

Comme le papier ne modélise pas de canal STE (article de 2005, le canal
STE vient de Chimier 2011), on met `enable_ste=False` pour cette
reproduction -- sinon on ne compare pas la même physique.

Vérifié : avec `w_f=1.1 µm`, `n0` (Sellmeier à 800 nm) ≈ 1.4533, le rayon de
Rayleigh `zf` ≈ 6.9 µm, et `t_p` ≈ 135.9 fs -- cohérent avec les 136 fs
donnés dans le texte. `Pin/Pcr` ≈ 3.26 pour 1.1 µJ, un régime bien
au-dessus du seuil d'autofocalisation, cohérent avec la clampe d'intensité
à ~5×10¹³ W/cm² rapportée dans le papier.

In [ ]:
PAPER_PARAMS = dict(
    # --- grille ---
    # L'article (Sec. III) demarre les simulations a d = 75 um AVANT le foyer
    # lineaire, et toutes ses figures (6/7/8/9/12/13) portent z = distance
    # depuis la face d'entree sur 0-150 um. On couvre donc exactement
    # z_sim in [-75, +75] um : begin=-75e-6, end=+75e-6.
    # R_factor=100 -> R_max=100 um alors que la largeur diffractee lineaire au
    # bout de la boite ne fait que w(d)=w_f*sqrt(1+(d/z_f)^2)=13.2 um : x7.5 de
    # marge pour le halo defocalise par le plasma, masque absorbant a 90 um.
    # Nr=3000 -> dr=33 nm, soit ~30 points dans w_f.
    Nz=6000, Nt=2048, Nr=3000,
    begin=-75e-6, end=75e-6,
    R_factor=100.0,
    save_stride=10,
    ckpt_every=200,
    verbose=True,
    # laser -- valeurs du papier (Sec. II-III)
    wavelength=800e-9,
    energy_uJ=1.1,      # Figs. 6, 7(b), 10, 12, 13
    w0=1.0e-6,          # w_f, Sec. III (z_f = pi w_f^2 n0/lam0 = 5.7 um le confirme ;
                        # Table I donne 1.1 um MESURE, mais les simus utilisent 1.0 um)
    delta_t=160e-15,    # FWHM 160 fs -> tp = 136 fs
    # matériau -- valeurs du papier (Sec. III)
    n2=3.54e-20,        # 3.54e-16 cm^2/W
    Ui_eV=9.0,
    meff_rel=0.64,
    tau_c=1e-14,        # determine par les mesures de transmission, Sec. IV.B
    tau_r=150e-15,      # Sec. III / experiences pump-probe
    rho_max=2.1e22,
    f_R=0.18, tau_d=32e-15, tau_s=12e-15,
    enable_ste=False,   # papier de 2005 : pas de canal STE
    # sonde (inutilisee pour cette verification, gardee pour compatibilite)
    lambda_probe=490e-9,
    # Cube (z,r,t) DESACTIVE : c'est 98 % du result.npz (8.3 Go sur 8.4 Go a
    # cette resolution) et aucune figure de l'article n'en a besoin -- elles
    # utilisent fluence_rz / rho_rz / Imax_z / absorbed_rz et les traces
    # on-axis PLEINE resolution (rho_onaxis_t, I_onaxis_t). Sans lui un run
    # fait ~150 Mo. Le cube ne sert qu'a fig10_dephasage et a l'export web.
    rho_t_stride=0,
    envelope="gaussian_focused",
)

PAPER_OUT_DIR = str(OUT_ROOT / "couairon2005_1p1uJ")
paper_result = load_scenario_npz(PAPER_OUT_DIR)
if paper_result is not None:
    print(f"[couairon2005_1p1uJ] déjà calculé -> {PAPER_OUT_DIR}/result.npz")
else:
    print(f"[couairon2005_1p1uJ] lancement -> {PAPER_OUT_DIR}")
    paper_result = run(out_dir=PAPER_OUT_DIR, **PAPER_PARAMS)

results["couairon2005_1p1uJ"] = paper_result

# z=57 um dans le repere de l'article (mesure depuis la face d'entree, cf.
# Fig. 7/10) <-> z_sim = 57 - 75 = -18 um dans notre repere centre sur le
# foyer lineaire (puisque la face d'entree est a z_sim = begin = -75 um).
Z_FIG10_UM = 57.0 - 75.0

# Le solveur travaille dans un repere centre sur le foyer lineaire (z=0), mais
# l'article mesure z depuis la face d'entree du zoom (z=0 = entree, Figs. 6/7/
# 9/10/12/13). Z_SHIFT_PAPER_UM ramene l'AFFICHAGE dans le repere de l'article
# (= -begin_um) ; z_target_um reste dans le repere du solveur (voir _z_um).
Z_SHIFT_PAPER_UM = -PAPER_PARAMS["begin"] * 1e6

# Bornes lues directement sur les Figs. 7/8/9/10 du papier (PhysRevB.71.125435) :
#   Fig. 8  z in [0, 150] um,   I   in [1e12, 1e14]  W/cm^2
#   Fig. 9  z in [0, 150] um,   rho in [1e18, 1e21]  cm^-3
#   Fig. 10 t in [-300, 300] fs, rho in [1e15, 1e21] cm^-3 (gauche), I in [1e12, 1e14] W/cm^2 (droite)
#   Fig. 7  z in [20, 120] um (panneau b, 1.1 uJ),   r in [-5, 5] um
plot_scenario_summary(
    paper_result, "couairon2005_1p1uJ",
    z_target_um=Z_FIG10_UM,          # Fig. 10 : z=57 um depuis l'entree
    rho_lines=(2e20, 4e20),          # Fig. 9 : clampage rapporte par le papier
    fluence_levels=(1.0, 2.0, 3.0),  # Fig. 7 : niveaux de contour du papier
    z_shift_um=Z_SHIFT_PAPER_UM,
    z_xlim=(0, 150), I_ylim=(1e12, 1e14), rho_ylim=(1e18, 1e21),
    trapped_ylim=(1e13, 1e21), trapped_ylim_I=(1e8, 1e14),
    fluence_xlim=(20, 120), fluence_rlim=(-5, 5),
    save=str(Path(PAPER_OUT_DIR) / "summary.png"))

# Params physiques partages par toutes les figures de cette section
# (figures_article.py) -- construits depuis PAPER_PARAMS pour rester coherents.
# sim/ et web/ sont deja sur sys.path (cellule d'imports)
from figures_article import Params as ArticleParams

paper_prm = ArticleParams(
    wavelength=PAPER_PARAMS["wavelength"],
    Ui_eV=PAPER_PARAMS["Ui_eV"],
    meff_rel=PAPER_PARAMS["meff_rel"],
    tau_c=PAPER_PARAMS["tau_c"],
    tau_r=PAPER_PARAMS["tau_r"],
    rho_max=PAPER_PARAMS["rho_max"],
    n2=PAPER_PARAMS["n2"],
    lambda_probe=PAPER_PARAMS["lambda_probe"],
)

### Fig. 2 : taux d'ionisation (aucune simulation requise)

Test le plus direct que le `W_PI` du solveur est bien celui de l'article :
c'est une pure évaluation de la formule de Keldysh générale, comparée à la
limite multiphotonique `W_MPI = σ₆ I⁶ ρ_at` (σ₆ = 9.6e-70 s⁻¹cm¹²/W⁶).

Points de contrôle chiffrés de l'article (Sec. III, p. 3), à I = 3.5e13 W/cm² :
γ = 1, `W_MPI` = 3.7e34 — tous deux reproduits exactement. Le `W_PI` du code
tombe à 4.3e32 contre 1.6e32 annoncé : un facteur 2.7, qui ne correspond qu'à
~15 % d'écart en intensité puisque `W_PI ~ I^K` avec K ≈ 6–8. L'article
signale lui-même une ambiguïté de convention sur β (« divisé par 4 » chez eux,
« divisé par 2 » chez Keldysh) ; aucune des deux ne redonne 1.6e32, donc le
résidu n'est pas tranchable sans la référence 42 et est laissé tel quel.

In [ ]:
from figures_article import fig2_ionization_rate

fig2_ionization_rate(paper_prm, I_min=1e12, I_max=1.5e14, I_marker=5e13,
                     ylim=(1e25, 1e35),
                     save=str(OUT_ROOT / "couairon2005_fig2_ionization_rate.png"));

### Comparaison directe aux figures du papier

- **Fig. 6/7(b)** : contours de fluence à 1, 2, 3 J/cm² (niveaux du papier,
  pas ceux par défaut) -- filament attendu entre ~25 et ~110 µm pour 1.1 µJ.
- **Fig. 9** : le papier rapporte un clampage de la densité électronique
  entre 2×10²⁰ et 4×10²⁰ cm⁻³ pour ces paramètres -- lignes de référence
  ajustées en conséquence (au lieu de 1e20/2e20 par défaut).

In [ ]:
# Fig. 7 de l'article : contours de fluence 1/2/3 J/cm2 + diametre FWHM,
# DEUX panneaux -- (a) 0.45 uJ et (b) 1.1 uJ, z in [20,120] um, r in [-5,5] um.
# Le run 0.45 uJ porte le meme out_dir que dans le balayage en energie plus
# bas, donc il est calcule une seule fois puis relu depuis le cache.
FIG7A_OUT_DIR = str(OUT_ROOT / "couairon2005_0.45uJ")
fig7a_result = load_scenario_npz(FIG7A_OUT_DIR)
if fig7a_result is None:
    print(f"[couairon2005_0.45uJ] lancement -> {FIG7A_OUT_DIR}")
    fig7a_result = run(out_dir=FIG7A_OUT_DIR, **{**PAPER_PARAMS, "energy_uJ": 0.45})
else:
    print("[couairon2005_0.45uJ] deja calcule -> chargement")

for _res, _lab, _f in ((fig7a_result, "(a) 0.45 µJ", "fig7a_fluence_045uJ.png"),
                       (results["couairon2005_1p1uJ"], "(b) 1.1 µJ", "fig7b_fluence_11uJ.png")):
    plot_fig7_fluence_contours(
        _res, levels=(1.0, 2.0, 3.0), label=f"Couairon 2005 {_lab}",
        z_shift_um=Z_SHIFT_PAPER_UM, xlim=(20, 120), rlim=(-5, 5),
        save=str(OUT_ROOT / _f));

# Test chiffré de la Fig. 7 (valeurs données dans le texte de l'article) :
#   0.45 µJ -> fluence > 1 J/cm² de 45 à 90 µm
#   1.1  µJ -> fluence > 1 J/cm² de 25 à 110 µm
fluence_level_extent(fig7a_result, label="0.45 µJ  (article: 45 -> 90 µm)",
                     z_shift_um=Z_SHIFT_PAPER_UM)
fluence_level_extent(results["couairon2005_1p1uJ"], label="1.1 µJ   (article: 25 -> 110 µm)",
                     z_shift_um=Z_SHIFT_PAPER_UM)

### Contrôle de santé : les autres chiffres de l'article

Les étendues de la Fig. 7 tombent juste, mais l'article donne deux autres
valeurs chiffrées qu'il faut vérifier séparément — l'intensité de clamping
(5 ± 0.5e13 W/cm², *décroissante* quand l'énergie baisse) et la densité
électronique (clampée entre 2e20 et 4e20 cm⁻³).

Cette cellule rappelle aussi quels interrupteurs ont **réellement** servi, lus
dans `params.json`. C'est le piège à connaître : `load_scenario_npz` relit un
`result.npz` existant sans regarder la version du code qui l'a produit, donc un
run en cache donne des chiffres identiques au digit près même après un
changement de solveur.

In [ ]:
for _name, _dir in (("0.45 µJ", FIG7A_OUT_DIR),
                    ("1.1 µJ",  PAPER_OUT_DIR)):
    _res = load_scenario_npz(_dir)
    if _res is not None:
        run_health_check(_res, out_dir=_dir, label=_name)
        print()

In [ ]:
plot_fig9_electron_density(
    {"couairon2005_1p1uJ": results["couairon2005_1p1uJ"]}, rho_lines=(2e20, 4e20),
    z_shift_um=Z_SHIFT_PAPER_UM,
    xlim=(0, 150), ylim=(1e18, 1e21),   # Fig. 9 du papier
    save=str(Path(PAPER_OUT_DIR) / "fig9_electron_density.png"));

# Fig. 10 de l'article : rho_e(t) AVEC avalanche (continu) vs SANS avalanche
# (tirets) + intensite (point-tirets), a z=57 um depuis l'entree.
# Deux points importants lus dans la legende de l'article :
#   - c'est un pulse de 1 uJ, PAS 1.1 uJ comme les Figs. 6/7(b)/12/13 ;
#   - la decomposition est avalanche ON/OFF, pas libres/pieges (l'article de
#     2005 n'a aucun canal STE, donc rho_s y est identiquement nul).
from figures_article import fig10_avalanche_vs_time

FIG10_OUT_DIR = str(OUT_ROOT / "couairon2005_1uJ")
fig10_result = load_scenario_npz(FIG10_OUT_DIR)
if fig10_result is None:
    print(f"[couairon2005_1uJ] lancement -> {FIG10_OUT_DIR}")
    fig10_result = run(out_dir=FIG10_OUT_DIR, **{**PAPER_PARAMS, "energy_uJ": 1.0})
else:
    print(f"[couairon2005_1uJ] deja calcule -> chargement")
results["couairon2005_1uJ"] = fig10_result

fig10_avalanche_vs_time(
    fig10_result, paper_prm, z_um=Z_FIG10_UM, z_shift_um=Z_SHIFT_PAPER_UM,
    xlim=(-300, 300), ylim=(1e13, 1e21), ylim_I=(1e8, 1e14),   # bornes de la Fig. 10
    save=str(Path(FIG10_OUT_DIR) / "fig10_avalanche_vs_time.png"));


### Fig. 13 : densité électronique vs z (trois variantes du modèle)

Contrairement aux Fig. 7/9/10 déjà reproduites ci-dessus, la Fig. 13 du
papier n'a pas d'équivalent direct dans les fonctions de tracé génériques
(section 0) : c'est une réintégration 0D balayée sur tout z (comme
`fig2_populations`, mais z par z au lieu d'un z fixé), comparant PI seul /
PI+recombinaison / PI+avalanche+recombinaison, avec le `ρ_e` réel du noyau
CUDA superposé en validation.

In [ ]:
# sim/ et web/ sont deja sur sys.path (cellule d'imports)
from figures_article import fig13_electron_density_vs_z

fig13_electron_density_vs_z(
    results["couairon2005_1p1uJ"], paper_prm,
    z_shift_um=Z_SHIFT_PAPER_UM,
    save=str(Path(PAPER_OUT_DIR) / "fig13_electron_density_vs_z.png"));

In [ ]:
# Fig. 12 du papier : pertes d'energie cumulees (fraction de U0), Plasma vs
# Photo vs combine -- E_MPI_z/E_plasma_z/E_total_z ne sont plus des zeros
# (voir filament_sim.py NonlinearOperator.loss_rates + Integrator._record :
# ce canal de diagnostic n'etait pas branche avant).
plot_fig12_energy_losses(
    results["couairon2005_1p1uJ"], label="Couairon 2005, 1.1 µJ",
    z_shift_um=Z_SHIFT_PAPER_UM, xlim=(0, 150), ylim=(1e-3, 1e0),
    save=str(Path(PAPER_OUT_DIR) / "fig12_energy_losses.png"));

### Fig. 8 / Fig. 9 du papier : balayage sur les six énergies

Le papier trace l'intensité crête (Fig. 8) et la densité électronique
on-axis (Fig. 9) vs z pour six énergies : 0.25, 0.45, 0.625, 0.85, 1.1 et
1.25 µJ. Boucle similaire à la section 4, réutilisant `PAPER_PARAMS`.

In [ ]:
COUAIRON_ENERGIES_UJ = [0.25, 0.45, 0.625, 0.85, 1.1, 1.25]
couairon_sweep = {}
pbar_couairon = tqdm(COUAIRON_ENERGIES_UJ, desc="Couairon 2005 -- balayage en énergie")
for e_uJ in pbar_couairon:
    name = f"couairon2005_{e_uJ:g}uJ"
    pbar_couairon.set_description(f"Couairon 2005: {e_uJ:g} µJ")
    out_dir = str(OUT_ROOT / name)
    cached = load_scenario_npz(out_dir)
    if cached is not None:
        pbar_couairon.write(f"[{name}] déjà calculé -> chargement")
        couairon_sweep[name] = cached
    else:
        pbar_couairon.write(f"[{name}] lancement -> {out_dir}")
        couairon_sweep[name] = run(out_dir=out_dir, **{**PAPER_PARAMS, "energy_uJ": e_uJ})

plot_fig8_peak_intensity(couairon_sweep, I_clamp=5e13, z_shift_um=Z_SHIFT_PAPER_UM,
                          xlim=(0, 150), ylim=(1e12, 1e14),   # Fig. 8 du papier
                          save=str(OUT_ROOT / "couairon2005_fig8_energy_sweep.png"));

In [ ]:
plot_fig9_electron_density(couairon_sweep, rho_lines=(2e20, 4e20), z_shift_um=Z_SHIFT_PAPER_UM,
                            xlim=(0, 150), ylim=(1e18, 1e21),   # Fig. 9 du papier
                            save=str(OUT_ROOT / "couairon2005_fig9_energy_sweep.png"));

## 1bis. Vérification croisée : Bulgakova, Stoian & Rosenfeld

*« Laser-induced modification of transparent crystals and glasses »* — leurs
Figs. 11 et 12, même matériau (silice) mais **jeu de paramètres différent** de
Couairon, ce qui en fait un second test indépendant du solveur.

Conditions (leur Sec. 5.2) : 800 nm, τ_L = 120 fs FWHM (soit τ_las = 100 fs en
demi-largeur 1/e du champ), E_in = 1 µJ, w = 0.9 µm, **foyer géométrique à 90 µm
sous la surface**. Silice : n_lat = 6.6e22 cm⁻³, k″ = 361 fs²/cm,
n₂ = 2.48e-16 cm²/W, E_g0 = 9 eV, f_R = 0.18, **m_r = 0.5 mₑ**, t_tr = 150 fs,
**ω₀τ_c = 3** (contre 23.6 chez Couairon).

Les écarts de modèle entre les deux articles sont listés dans
`ANALYSE_CRITIQUE.md` §6 — le principal étant que leur gap est **corrigé
pondéromotivement** (E_g = E_g0 + U_p) alors que Couairon garde U_i = 9 eV fixe.
Le solveur suit Couairon sur ce point, donc un écart est attendu à haute
intensité.

In [ ]:
BULGAKOVA_PARAMS = dict(
    # --- grille ---
    # Foyer geometrique a 90 um sous la surface -> begin = -90 um ; leurs
    # figures vont de z=0 (surface) a 150 um, d'ou end = +60 um.
    Nz=6000, Nt=2048, Nr=3000,
    begin=-90e-6, end=60e-6,
    R_factor=110.0,
    save_stride=10, ckpt_every=200, verbose=True,
    # --- laser (leur Sec. 5.2) ---
    wavelength=800e-9,
    energy_uJ=1.0,
    w0=0.9e-6,
    delta_t=120e-15,      # FWHM ; tau_las = FWHM/sqrt(2 ln2) = 102 fs ~ leurs 100 fs
    # --- silice, LEURS valeurs (differentes de Couairon) ---
    n2=2.48e-20,          # 2.48e-16 cm^2/W  (Couairon : 3.54e-16)
    Ui_eV=9.0,            # E_g0
    meff_rel=0.5,         # m_r = 0.5 m_e    (Couairon : 0.64)
    tau_c=3.0 / (2 * np.pi * 299792458.0 / 800e-9),   # omega0*tau_c = 3 (Couairon : 23.6)
    tau_r=150e-15,        # t_tr, piegeage
    rho_max=6.6e22,       # n_lat            (Couairon : 2.1e22)
    f_R=0.18, tau_d=32e-15, tau_s=12e-15,
    enable_ste=False,     # leur eq. (21) n'a pas de canal STE explicite
    lambda_probe=490e-9,
    rho_t_stride=0,
    envelope="gaussian_focused",
    # --- enregistrements specifiques aux Figs. 11 et 12 ---
    rho_snapshot_t_fs=50.0,                        # Fig. 11d : rho a +50 fs
    absorb_time_bins_fs=(-100, -50, 0, 50, 100),   # Fig. 12  : 4 tranches
)

BULG_OUT_DIR = str(OUT_ROOT / "bulgakova_1uJ")
bulg_result = load_scenario_npz(BULG_OUT_DIR)
if bulg_result is None:
    print(f"[bulgakova_1uJ] lancement -> {BULG_OUT_DIR}")
    bulg_result = run(out_dir=BULG_OUT_DIR, **BULGAKOVA_PARAMS)
else:
    print("[bulgakova_1uJ] deja calcule -> chargement")
results["bulgakova_1uJ"] = bulg_result

# z=0 = surface de l'echantillon, foyer geometrique a +90 um
Z_SHIFT_BULG_UM = -BULGAKOVA_PARAMS["begin"] * 1e6

In [ ]:
from figures_article import fig11_bulgakova, fig12_bulgakova

fig11_bulgakova(bulg_result, z_shift_um=Z_SHIFT_BULG_UM, focus_um=90.0,
                save=str(Path(BULG_OUT_DIR) / "fig11.png"));

In [ ]:
fig12_bulgakova(bulg_result, z_shift_um=Z_SHIFT_BULG_UM,
                save=str(Path(BULG_OUT_DIR) / "fig12.png"));

### Export : toutes les figures dans un seul PDF (titres et légendes en anglais)

`save_all_figures` parcourt `results` et écrit chaque figure disponible dans un
unique PDF multi-pages. Ce qui manque est simplement sauté avec une note, donc
la cellule est utilisable après la seule section 1 comme après tout le notebook.

In [ ]:
from figures_article import save_all_figures

save_all_figures(results, OUT_ROOT / "figures_all.pdf", paper_prm,
                 paper_shift_um=Z_SHIFT_PAPER_UM, z_fig10_um=Z_FIG10_UM);

## 2. Paramètres de base (TON expérience -- communs à toutes les configurations)

Reprend les valeurs numériques de Couairon et al. 2005 (grille, laser,
matériau -- section 1), avec `enable_ste=True` (ton choix : canal STE actif,
absent du modèle du papier). Grille : `Nz=6000` (dz ≈ 42 nm sur la boîte
250 µm), `Nt=2048`, `Nr=3000` points radiaux Hankel (`R_factor=100`, voir le
commentaire dans la cellule ci-dessous pour le calcul). Réduis ces valeurs
pour un premier test rapide (quelques minutes), remets-les pour la
production.

In [ ]:
BASE_PARAMS = dict(
    # grille -- R_factor recalcule pour w0=1.1 um (PAS un R_factor generique
    # herite d'un autre w0) : a z=+175 um (bout le plus large de la boite),
    # la largeur diffractee lineaire est w(z)=w_f*sqrt(1+(z/zf)^2)~28 um
    # (zf=6.9 um a 800 nm). R_factor=100 -> R_max=110 um, masque absorbant a
    # 0.9*R_max=99 um, marge x4 sur le halo defocalise par le plasma apres
    # le foyer non-lineaire. Nr=3000 pour garder ~30 points dans w0.
    Nz=6000, Nt=2048, Nr=3000,
    begin=-75e-6, end=175e-6,
    R_factor=100.0,
    save_stride=10,
    ckpt_every=200,
    verbose=True,
    # laser -- valeurs de Couairon et al., PRB 71, 125435 (2005), Sec. II-III
    wavelength=800e-9,
    energy_uJ=1.1,      # Figs. 6, 7(b), 10, 12, 13 du papier
    w0=1.1e-6,          # w_f, Table I (objectif 20x, NA=0.5)
    delta_t=160e-15,    # FWHM 160 fs -> tp = 136 fs
    # matériau -- valeurs du papier (Sec. III)
    n2=3.54e-20,        # 3.54e-16 cm^2/W
    Ui_eV=9.0,
    meff_rel=0.64,
    tau_c=1e-14,        # determine par les mesures de transmission, Sec. IV.B
    tau_r=150e-15,      # Sec. III / experiences pump-probe
    rho_max=2.1e22,
    f_R=0.18, tau_d=32e-15, tau_s=12e-15,
    enable_ste=True,    # ton choix : canal STE actif (absent du papier de 2005)
    # sonde (pour le futur export Abel)
    lambda_probe=490e-9,
    rho_t_stride=5,
    # Le cube (z,r,t) sert a fig10_dephasage et a l'export web. Il domine
    # la taille du fichier, donc on le sous-echantillonne aussi en rayon :
    # le filament fait ~100 points de large sur 3000, rho_r_stride=8 suffit
    # largement et divise le cube par 8.
    rho_r_stride=8,
    envelope="gaussian_focused",
)

print(f"Nz={BASE_PARAMS['Nz']}, begin={BASE_PARAMS['begin']*1e6:+.1f} µm, "
      f"dz={(BASE_PARAMS['end']-BASE_PARAMS['begin'])/BASE_PARAMS['Nz']*1e9:.1f} nm")

## 3. Interrupteurs interactifs

Coche/décoche les termes, clique sur *Lancer* pour une simulation unique avec
cette configuration (stockée dans `manual_result` / `manual_out_dir`).

In [ ]:
FIELD_LABELS = {
    "enable_kerr_instantaneous":  "Kerr instantané",
    "enable_kerr_raman":          "Kerr Raman (retardé)",
    "enable_self_steepening":     "Self-steepening (T-hat)",
    "enable_photoionization_loss":"Perte par photoionisation (champ)",
    "enable_plasma_defocusing":   "Défocalisation plasma (phase)",
    "enable_plasma_absorption":   "Absorption plasma (Bremsstrahlung inverse)",
}
CARRIER_LABELS = {
    "enable_avalanche":      "Ionisation par avalanche (porteurs)",
    "enable_recombination":  "Recombinaison (porteurs)",
    "enable_ste":            "Canal excitons auto-piégés (STE)",
}

_checkboxes = {}
boxes = []
for key, label in {**FIELD_LABELS, **CARRIER_LABELS}.items():
    cb = widgets.Checkbox(value=True, description=label, indent=False,
                           layout=widgets.Layout(width="420px"))
    _checkboxes[key] = cb
    boxes.append(cb)

run_name = widgets.Text(value="manual_run", description="out_dir:")
run_button = widgets.Button(description="Lancer la simulation", button_style="primary")
run_status = widgets.Output()

def toggles_from_widgets():
    return {k: cb.value for k, cb in _checkboxes.items()}

manual_result = None
manual_out_dir = None

def _on_run_clicked(_):
    global manual_result, manual_out_dir
    with run_status:
        run_status.clear_output()
        overrides = toggles_from_widgets()
        manual_out_dir = str(OUT_ROOT / run_name.value)
        print("Toggles:", overrides)
        print("out_dir:", manual_out_dir)
        manual_result = run(out_dir=manual_out_dir, **{**BASE_PARAMS, **overrides})
        print("Terminé.")

run_button.on_click(_on_run_clicked)

display(widgets.VBox([widgets.HTML("<b>Champ (éq. 3)</b>")]
                      + boxes[:len(FIELD_LABELS)]
                      + [widgets.HTML("<b>Porteurs (éq. 6-7)</b>")]
                      + boxes[len(FIELD_LABELS):]
                      + [widgets.HBox([run_name, run_button]), run_status]))

## 4. Validation initiale : lancer `full` seul d'abord

Avant de lancer les 7 autres scénarios (qui peuvent prendre du temps sur la
grille de production), on lance uniquement `full` (tout activé) et on
vérifie que ça a une allure physique cohérente :
- la figure compacte 2x2 (`plot_scenario_summary`, section 2) ;
- `fig2_populations` (`sim/figures_article.py`) : réintégration 0D des
  populations électroniques (MPI seul / + avalanche / + piégeage-STE) à
  partir de `I(t)` sauvegardé, superposée au `ρ_e` réellement calculé par
  le noyau CUDA (points rouges) -- si les deux ne coïncident pas, il y a
  un problème entre le solveur et l'équation de taux ;
- `fig10_dephasage` : déphasage sonde 490 nm décomposé en Kerr croisé /
  Drude / STE, à comparer à la Fig. 10 de Mao et al., Appl. Phys. A 79,
  1695 (2004).

**Si ça ne matche pas l'article, corrige avant de lancer la boucle
complète** -- pas la peine de brûler du temps GPU sur 7 scénarios si
`full` est déjà faux. `full` reste ensuite en cache : la boucle d'ablation
(section 4) ne le relance pas.

In [ ]:
FULL_OUT_DIR = str(OUT_ROOT / "full")
full_result = load_scenario_npz(FULL_OUT_DIR)
if full_result is not None:
    print(f"[full] déjà calculé -> chargement de {FULL_OUT_DIR}/result.npz")
else:
    print(f"[full] lancement -> {FULL_OUT_DIR}")
    full_result = run(out_dir=FULL_OUT_DIR, **BASE_PARAMS)

results["full"] = full_result
plot_scenario_summary(results["full"], "full", save=str(Path(FULL_OUT_DIR) / "summary.png"))

In [ ]:
# sim/ et web/ sont deja sur sys.path (cellule d'imports)
from figures_article import fig2_populations, fig10_dephasage, Params as ArticleParams

# Mêmes paramètres physiques que BASE_PARAMS, pour que la réintégration 0D
# de figures_article.py utilise exactement le même matériau que le solveur.
# tau_c (et Us_eV/meff_rel, par robustesse si BASE_PARAMS change un jour)
# doivent être transmis explicitement : Params a pour défaut tau_c=1.7e-15 s,
# alors que BASE_PARAMS utilise 1e-14 s -- un oubli ici fait diverger
# sigma_omega (donc beta_g/beta_s) d'un facteur ~5.5x et fausse la
# réintégration 0D de fig2_populations/fig10_dephasage par rapport au
# vrai noyau CUDA.
article_prm = ArticleParams(
    wavelength=BASE_PARAMS["wavelength"],
    Ui_eV=BASE_PARAMS["Ui_eV"],
    Us_eV=BASE_PARAMS.get("Us_eV", 6.0),
    meff_rel=BASE_PARAMS["meff_rel"],
    tau_c=BASE_PARAMS["tau_c"],
    rho_max=BASE_PARAMS["rho_max"],
    tau_r=BASE_PARAMS["tau_r"],
    n2=BASE_PARAMS["n2"],
    lambda_probe=BASE_PARAMS["lambda_probe"],
)

fig2_populations(results["full"], article_prm, save=str(Path(FULL_OUT_DIR) / "fig2_populations.png"));

In [ ]:
fig10_dephasage(results["full"], article_prm, band_um=10.0, save=str(Path(FULL_OUT_DIR) / "fig10_dephasage.png"));

## 5. Boucle d'ablation automatique (scénarios restants)

Une configuration par terme désactivé isolément (`no_<terme>`) + une
configuration tout-linéaire (`linear_only`, les six désactivés). `full` a
déjà été validé et calculé en section 3, donc cette boucle ne le relance
pas. Les résultats déjà présents sur disque (`result.npz` + `params.json`)
sont rechargés au lieu d'être relancés, pour pouvoir reprendre la boucle
après interruption.

**Tracé à la volée** : dès qu'un scénario se termine (ou est rechargé du
cache), sa figure compacte (`plot_scenario_summary`) s'affiche
immédiatement, avant de passer au suivant.

In [ ]:
def build_scenarios():
    scenarios = {"full": {}}
    for f in FIELD_TOGGLES:
        scenarios[f"no_{f.replace('enable_', '')}"] = {f: False}
    scenarios["linear_only"] = {f: False for f in FIELD_TOGGLES}
    return scenarios

SCENARIOS = build_scenarios()
for name, overrides in SCENARIOS.items():
    print(f"{name:28s} {overrides}")

In [ ]:
# Barre de progression sur les scénarios (extérieure) : indique lequel tourne
# et combien il en reste. Chaque scénario a EN PLUS sa propre barre de
# progression pas-à-pas (barre "Filamentation" affichée par
# Integrator.propagate() dans filament_sim.py), donc tu vois à la fois
# la progression globale (scénario X/8) et le détail (pas z Y/Nz, ETA).
pbar_scenarios = tqdm(list(SCENARIOS.items()), desc="Scénarios")
for name, overrides in pbar_scenarios:
    pbar_scenarios.set_description(f"Scénario: {name}")
    if name in results:
        pbar_scenarios.write(f"[{name}] déjà en mémoire (validé en section 3)")
        continue
    out_dir = str(OUT_ROOT / name)
    cached = load_scenario_npz(out_dir)
    if cached is not None:
        pbar_scenarios.write(f"[{name}] déjà calculé -> chargement de {out_dir}/result.npz")
        results[name] = cached
    else:
        pbar_scenarios.write(f"[{name}] lancement -> {out_dir}  (overrides={overrides})")
        results[name] = run(out_dir=out_dir, **{**BASE_PARAMS, **overrides})

    # Tracé à la volée -- voir section 2 pour plot_scenario_summary
    plot_scenario_summary(results[name], name, save=str(OUT_ROOT / name / "summary.png"))

pbar_scenarios.set_description("Scénarios")
print("Scénarios disponibles:", list(results.keys()))

## 6. Comparaison superposée des scénarios

Les mêmes quatre styles de figure que la boucle ci-dessus, mais superposés
sur tous les scénarios pour comparer l'effet de chaque terme (fonctions
publiques définies en section 2).

In [ ]:
# scenario_names=list(SCENARIOS.keys()) : results contient aussi la
# reproduction Couairon 2005 (section 1), a ne pas melanger avec tes
# 8 scenarios d'ablation ici.
plot_fig8_peak_intensity(results, scenario_names=list(SCENARIOS.keys()),
                          save=str(OUT_ROOT / "comparison_fig8_peak_intensity.png"));

In [ ]:
plot_fig9_electron_density(results, scenario_names=list(SCENARIOS.keys()), nc_probe_cm3=4.6e21,
                            save=str(OUT_ROOT / "comparison_fig9_electron_density.png"));

In [ ]:
plot_fig13_free_vs_trapped(results["full"], label="full",
                            save=str(OUT_ROOT / "comparison_fig13_free_vs_trapped.png"));

In [ ]:
plot_fig7_fluence_contours(results["full"], label="full",
                            save=str(OUT_ROOT / "comparison_fig7_fluence_contours.png"));

In [ ]:
plot_fig12_energy_losses(results["full"], label="full",
                          save=str(OUT_ROOT / "comparison_fig12_energy_losses.png"));

## 7. Tableau récapitulatif comparatif

In [ ]:
def summarize(results):
    rows = []
    for name, res in results.items():
        z_um = _z_um(res)
        Imax = res["Imax_z"]
        r = np.asarray(res["r"]); i_axis = int(np.argmin(np.abs(r)))
        rho_onaxis_max = float(res["rho_rz"][:, i_axis].max())
        i_peak_idx = int(np.argmax(Imax))
        rows.append(dict(
            scenario=name,
            I_peak_Wcm2=float(Imax[i_peak_idx]),
            z_at_Ipeak_um=float(z_um[i_peak_idx]),
            rho_e_onaxis_max_cm3=rho_onaxis_max,
        ))
    return rows

# on ne resume que TES 8 scenarios d'ablation ici (pas la reproduction
# Couairon 2005 de la section 1, qui vit dans le meme dict `results`)
for row in summarize({k: results[k] for k in SCENARIOS if k in results}):
    print(f"{row['scenario']:28s}  I_peak={row['I_peak_Wcm2']:.3e} W/cm² @ z={row['z_at_Ipeak_um']:+7.1f} µm"
          f"   ρ_e,max={row['rho_e_onaxis_max_cm3']:.3e} cm⁻³")

## 8. Export vers la page web Abel-transform

`web/abel_phase_explorer.py` lit `result.npz` + `params.json` d'un ou plusieurs
scénarios (produits par la boucle ci-dessus, qui écrit déjà `params.json`
via `Integrator._dump_params`) et construit une page HTML autonome avec :
- un sélecteur de scénario,
- des cases à cocher par canal de Δn (Drude / Kerr / STE / thermique),
- le panneau expérience (si `RAW_DIR` est fourni et contient les npz bruts),
recalculant le déphasage en direct dans le navigateur (somme linéaire des
canaux déjà transformés par Abel côté Python, donc pas de recalcul FFT côté
JS).

In [ ]:
# sim/ et web/ sont deja sur sys.path (cellule d'imports)
from abel_phase_explorer import build_explorer_html

# uniquement TES 8 scenarios ici -- la reproduction Couairon 2005 (800 nm)
# n'a pas de sens dans l'explorateur Abel a 490 nm de TON experience
sim_dirs = {name: str(OUT_ROOT / name) for name in SCENARIOS if name in results}
build_explorer_html(
    sim_dirs=sim_dirs,
    save="abel_phase_explorer.html",
    raw_dir=None,  # mets le chemin de tes npz bruts expérimentaux pour activer le panneau expérience
)
print("-> ouvre abel_phase_explorer.html dans un navigateur")

## 9. Filamentation à longue distance (0 -> 3 mm, puissance plus forte)

`BASE_PARAMS` (section 2) couvre `z in [-75, +175] um`, assez pour voir le
premier collapse (Fig. 8 ci-dessus) mais pas assez pour voir plusieurs cycles
de focalisation/défocalisation se former : la boîte se termine juste après le
premier foyer non-linéaire.

Ici on répète la même physique (mêmes toggles, même matériau) mais avec :
- une boîte 17x plus longue (`begin=0`, `end=3000e-6`, face d'entrée à z=0),
  `Nz` rescalé pour garder le même pas `dz` que `BASE_PARAMS` (sinon les
  cycles rapprochés seraient sous-échantillonnés, cf. le "wiggle" de la Fig. 7
  pleine boîte de la section 4) ;
- une énergie plus forte (`P_in/P_cr` ~10.4 au lieu de ~3.5), pour laisser
  assez de réservoir pour plusieurs cycles avant épuisement ;
- **PAS** un `R_factor`/`Nr` mis à l'échelle de la même façon que `Nz`. Le
  raisonnement naïf ("garder le même dr sur toute la largeur diffractée
  linéairement à 3 mm") demanderait `Nr ~ 55000`, ce qui est impossible avec
  ce solveur : `grids.py` construit une matrice de transformée de Hankel
  quasi-discrète pleine `Nr x Nr` (`Y` dans `build_grids`), donc la mémoire
  et le coût par pas croissent en `Nr^2` -- 24 Go rien que pour cette
  matrice à `Nr=55000`. La bonne raison physique de ne pas le faire : un
  faisceau bien au-dessus de `P_cr` reste confiné près de l'axe par le
  clampage d'intensité à chaque cycle, contrairement à un faisceau qui
  diffracte librement sur toute la boîte -- donc on n'a besoin de place que
  pour le halo de réservoir rejeté entre deux cycles, pas pour la largeur
  diffractée linéaire à 3 mm. `R_factor` est monté à 150 (au lieu de 100,
  `Nr=4500` pour garder le même `dr` que `BASE_PARAMS`) : marge
  supplémentaire par cycle, matrice `Y` encore parfaitement gérable.

Vérifier `absorbed_rz` au bord (`r=R_max`) une fois le run terminé : une
perte d'énergie non négligeable là-bas est le signal qu'il faut remonter
`R_factor` davantage, par petits pas, plutôt que de sauter direct à
l'estimation linéaire.

Coût attendu : `Nz` x27 et `Nr` x1.5 par rapport à `full` (BASE_PARAMS)
-- grossièrement un facteur ~27-40x le temps de calcul de `full`, à faire
tourner en conséquence (nuit / plusieurs heures de GPU). Le cube `(z,r,t)`
est désactivé (`rho_t_stride=0`) pour ne pas produire un `result.npz` de
plusieurs Go pour rien : aucune figure de cette section n'en a besoin.

In [ ]:
LONGBOX_PARAMS = dict(BASE_PARAMS)
LONGBOX_PARAMS.update(
    begin=0.0,                  # face d'entree, au lieu de -75e-6 (foyer lineaire)
    end=3000e-6,                # au lieu de 175e-6 -- 3 mm pour loger plusieurs cycles
    Nz=72000,                   # (3000e-6 - 0.0) / 41.7e-9 -- meme dz que BASE_PARAMS
    R_factor=150.0,             # au lieu de 100.0 -- plus de marge radiale par cycle
    Nr=4500,                    # 150*w0 / 36.7e-9 -- meme dr que BASE_PARAMS, PAS le
                                 # ~55000 qu'exigerait une marge de diffraction lineaire
                                 # (matrice de Hankel Nr x Nr dense -- voir texte ci-dessus)
    energy_uJ=3.3,               # 3x BASE_PARAMS -- P_in/P_cr ~10.4 au lieu de ~3.5
    ckpt_every=1000,             # moins de checkpoints sur 72000 pas
    rho_t_stride=0,              # cube (z,r,t) desactive -- inutile ici, dominerait
                                 # sinon la taille du fichier de sortie
)

print(f"Nz={LONGBOX_PARAMS['Nz']}, Nr={LONGBOX_PARAMS['Nr']}, "
      f"dz={(LONGBOX_PARAMS['end']-LONGBOX_PARAMS['begin'])/LONGBOX_PARAMS['Nz']*1e9:.1f} nm, "
      f"dr={LONGBOX_PARAMS['R_factor']*LONGBOX_PARAMS['w0']/LONGBOX_PARAMS['Nr']*1e9:.1f} nm, "
      f"energy={LONGBOX_PARAMS['energy_uJ']} uJ")

LONGBOX_OUT_DIR = str(OUT_ROOT / "longbox_3mm")
longbox_result = load_scenario_npz(LONGBOX_OUT_DIR)
if longbox_result is not None:
    print(f"[longbox_3mm] deja calcule -> chargement de {LONGBOX_OUT_DIR}/result.npz")
else:
    print(f"[longbox_3mm] lancement -> {LONGBOX_OUT_DIR}  (peut prendre plusieurs heures)")
    longbox_result = run(out_dir=LONGBOX_OUT_DIR, **LONGBOX_PARAMS)

results["longbox_3mm"] = longbox_result

In [ ]:
# Fraction d'energie perdue au bord absorbant (r=R_max) : si ce n'est pas
# negligeable, R_factor est trop petit pour ce run et doit etre augmente.
r_um = _r_um(longbox_result)
i_edge = np.argmax(r_um)  # dernier point radial = bord de la boite
edge_frac = float(np.max(longbox_result["absorbed_rz"][:, i_edge]) /
                   max(float(np.max(longbox_result["absorbed_rz"])), 1e-30))
print(f"Perte max au bord (r={r_um[i_edge]:.0f} um) relative a la perte max sur toute la boite : "
      f"{edge_frac*100:.1f} %  (regle empirique : si > quelques %, remonter R_factor)")

In [ ]:
plot_fig8_peak_intensity({"longbox_3mm": longbox_result}, xlim=(0, 3000),
                          save=str(Path(LONGBOX_OUT_DIR) / "fig8_peak_intensity_3mm.png"));

In [ ]:
plot_fig7_fluence_contours(longbox_result, levels=(1.0, 2.0, 3.0), label="longbox_3mm",
                            save=str(Path(LONGBOX_OUT_DIR) / "fig7_fluence_contours_3mm.png"));

### Diagnostic : compter les cycles

Chaque maximum local de la courbe d'intensité crête au-dessus de `I_clamp`,
au-dela du premier (celui deja vu Fig. 8), est un cycle de refocalisation
supplementaire. L'espacement croissant entre maxima successifs (si le
reservoir s'epuise progressivement) est la signature attendue.

In [ ]:
from scipy.signal import find_peaks

I_clamp = 5e13
z_um = _z_um(longbox_result)
peaks_idx, _ = find_peaks(longbox_result["Imax_z"], height=I_clamp)
print(f"{len(peaks_idx)} maximum(s) local(aux) au-dessus de I_clamp detecte(s) :")
for i, ip in enumerate(peaks_idx):
    print(f"  cycle {i+1}: z = {z_um[ip]:7.1f} um   I_peak = {longbox_result['Imax_z'][ip]:.3e} W/cm2")
if len(peaks_idx) >= 2:
    spacings = np.diff(z_um[peaks_idx])
    print("Espacements successifs (um):", np.round(spacings, 1))